# W&B Sweep — DQN
Búsqueda de hiperparámetros para el agente DQN en el ambiente Simple (CSTR).
- Método: Random Search
- Proyecto W&B: `Tesis_DQN`
- Arquitectura: Simple (CTRL únicamente)

## 1. Instalación e Imports

In [ ]:
import os
import random
import numpy as np
import torch
import wandb
import sys

In [ ]:
# Clonar desde Github:
!git clone https://github.com/valeriaeskenazi/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning.git
PROJECT_PATH = '/content/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning/Version_4'
sys.path.append(PROJECT_PATH)

fatal: destination path 'Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning' already exists and is not an empty directory.


In [ ]:
from Environment.Simulation_Env.Reactor_CSTR import CSTRSimulator
from Environment.PIDControlEnv_simple import PIDControlEnv_Simple
from Agente.DQN.train_DQN import DQNTrainer
from Aux.Plots import SimplePlotter, print_summary

print('Imports completados')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

Imports completados
PyTorch: 2.10.0+cu128
Device: CUDA


In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Tue Mar  3 01:35:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             48W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Login W&B

In [ ]:
!pip install wandb --quiet

In [ ]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ve326684 (ve326684-universidad-ort-uruguay) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Configuración del Sweep

In [ ]:
WANDB_TEAM    = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_DQN'

sweep_config = {
    'name':   'dqn_cstr_random_search',
    'method': 'random',

    'metric': {
        'name': 'eval_reward',
        'goal': 'maximize'
    },

    'parameters': {

        # ============ AMBIENTE ============
        'max_time_detector': {'values': [15, 30, 60]},
        'max_steps':         {'values': [20, 50, 100]},
        'reward_dead_band':  {'values': [0.01, 0.02, 0.05]},
        'delta_percent_ctrl':{'values': [0.05, 0.1, 0.2, 0.3]},

        # Reward weights — combinaciones predefinidas
        'reward_weights_idx': {
            'values': [0, 1, 2, 3]  # índice a la lista definida en sweep_run()
        },

        # ============ CRITERIOS DE ESTABILIDAD ============
        'error_increase_tolerance': {'values': [1.2, 1.5, 2.0]},
        'max_sign_changes_ratio':   {'values': [0.1, 0.2, 0.3]},
        'max_abrupt_change_ratio':  {'values': [0.03, 0.05, 0.1]},
        'abrupt_change_threshold':  {'values': [0.2, 0.3, 0.5]},

        # ============ AGENTE DQN ============
        'hidden_dims_idx':    {'values': [0, 1, 2, 3]},  # índice a lista en sweep_run()
        'lr':                 {'values': [0.00001, 0.0001, 0.001]},
        'epsilon_decay':      {'values': [0.999, 0.9995, 0.9999]},
        'target_update_freq': {'values': [50, 100, 200]},
        'batch_size':         {'values': [32, 64, 128]},
        'buffer_type':        {'values': ['simple', 'priority']},
        'buffer_size':        {'values': [5000, 10000, 50000]},
    }
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_TEAM)
print(f'Sweep creado: {sweep_id}')

Create sweep with ID: 9yv5itwu
Sweep URL: https://wandb.ai/ve326684-universidad-ort-uruguay/Tesis_DQN/sweeps/9yv5itwu
Sweep creado: 9yv5itwu


## 4. Función de Entrenamiento

In [ ]:
# ============ LISTAS DE OPCIONES PREDEFINIDAS ============

REWARD_WEIGHTS_OPTIONS = [
    {'error': 1.0, 'tiempo': 0.3, 'overshoot': 0.2, 'energy': 0.1},   # 0: balanceado (default)
    {'error': 2.0, 'tiempo': 0.1, 'overshoot': 0.5, 'energy': 0.1},   # 1: foco en error y overshoot
    {'error': 1.0, 'tiempo': 0.5, 'overshoot': 0.1, 'energy': 0.5},   # 2: foco en tiempo y energía
    {'error': 3.0, 'tiempo': 0.1, 'overshoot': 0.1, 'energy': 0.05},  # 3: solo error importa
]

HIDDEN_DIMS_OPTIONS = [
    (64, 32),
    (128, 64),
    (128, 128, 64),
    (256, 128, 64),
]

# ============ FIJOS PARA TODOS LOS RUNS ============
SEED             = 42
N_EPISODES       = 300
EVAL_FREQUENCY   = 50
EARLY_STOPPING_PATIENCE   = 10
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
N_MANIPULABLE_VARS = 2
MANIPULABLE_RANGES = [(300, 420), (99.5, 102)]
VAR_NAMES          = ['T (K)', 'V (m³)']
DT                 = 1.0
DEVICE             = 'cuda' if torch.cuda.is_available() else 'cpu'


def sweep_run():
    # -------- Inicializar run --------
    wandb.init()
    cfg = wandb.config

    # -------- Reproducibilidad --------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    wandb.config.update({'seed': SEED}, allow_val_change=True)

    # -------- Resolver índices --------
    reward_weights = REWARD_WEIGHTS_OPTIONS[cfg.reward_weights_idx]
    hidden_dims    = HIDDEN_DIMS_OPTIONS[cfg.hidden_dims_idx]

    # Loggear valores reales (no índices) para legibilidad en W&B
    wandb.config.update({
        'reward_weights': str(reward_weights),
        'hidden_dims':    str(hidden_dims),
    }, allow_val_change=True)

    # -------- Configurar CSTR --------
    cstr = CSTRSimulator(
        dt=DT,
        control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
    )

    # -------- Construir config del trainer --------
    trainer_config = {
        # === AMBIENTE ===
        'env_config': {
            'architecture':    'simple',
            'env_type':        'simulation',
            'n_manipulable_vars': N_MANIPULABLE_VARS,
            'manipulable_ranges': MANIPULABLE_RANGES,
            'manipulable_setpoints': None,  # random en cada episodio
            'dt_usuario':      DT,
            'max_steps':       cfg.max_steps,
            'max_time_detector': cfg.max_time_detector,
            'reward_dead_band':  cfg.reward_dead_band,
            'delta_percent_ctrl': cfg.delta_percent_ctrl,
            'reward_weights':  reward_weights,
            'pid_limits': [
                (0.01, 5.0),
                (0.001, 1.0),
                (0.0001,   1.0)
            ],
            'agent_controller_config': {'agent_type': 'discrete'},
            'env_type_config': {
                'dt': DT,
                'control_limits': (MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
            },
            # Criterios de estabilidad
            'stability_config': {
                'error_increase_tolerance': cfg.error_increase_tolerance,
                'max_sign_changes_ratio':   cfg.max_sign_changes_ratio,
                'max_abrupt_change_ratio':  cfg.max_abrupt_change_ratio,
                'abrupt_change_threshold':  cfg.abrupt_change_threshold,
            },
        },

        # === AGENTE DQN ===
        'agent_ctrl_config': {
            'state_dim':          N_MANIPULABLE_VARS * 5,  # 5 features por variable
            'action_dim':         7,
            'n_vars':             N_MANIPULABLE_VARS,
            'hidden_dims':        hidden_dims,
            'lr':                 cfg.lr,
            'gamma':              0.99,
            'epsilon_start':      1.0,
            'epsilon_min':        0.01,
            'epsilon_decay':      cfg.epsilon_decay,
            'batch_size':         cfg.batch_size,
            'target_update_freq': cfg.target_update_freq,
            'buffer_type':        cfg.buffer_type,
            'buffer_size':        cfg.buffer_size,
            'device':             DEVICE,
            'seed':               SEED,
        },

        # === ENTRENAMIENTO ===
        'n_episodes':          N_EPISODES,
        'eval_frequency':      EVAL_FREQUENCY,
        'save_frequency':      9999,  # no guardar checkpoints en sweep
        'log_frequency':       50,
        'checkpoint_dir':      f'checkpoints/dqn_{wandb.run.name}',
        'early_stopping_patience':      EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta_pct': EARLY_STOPPING_MIN_DELTA_PCT,
        'use_wandb': True,
    }

    # -------- Conectar CSTR al ambiente --------
    # Se hace después de crear el trainer para acceder al proceso
    trainer = DQNTrainer(trainer_config)
    trainer.env.proceso.connect_external_process(cstr)

    # -------- Entrenar --------
    trainer.train()

    # -------- Métricas finales del run --------
    wandb.log({
        'final_eval_reward':  trainer.best_reward,
        'total_episodes':     len(trainer.episode_rewards),
        'final_epsilon':      trainer.epsilons[-1] if trainer.epsilons else 0,
        'final_reward_mean10': np.mean(trainer.episode_rewards[-10:]),
        'final_energy_mean10': np.mean(trainer.episode_energies[-10:]),
        'final_overshoot_mean10': np.mean(trainer.episode_max_overshoots[-10:]),
    })

    print(f'Run completado: {wandb.run.name}')
    wandb.finish()


## 5. Lanzar Sweep

In [ ]:
wandb.agent(sweep_id, function=sweep_run, count=30)

wandb: Agent Starting Run: 7k2tfgub with config:
wandb: 	abrupt_change_threshold: 0.2
wandb: 	batch_size: 128
wandb: 	buffer_size: 50000
wandb: 	buffer_type: simple
wandb: 	delta_percent_ctrl: 0.05
wandb: 	epsilon_decay: 0.9995
wandb: 	error_increase_tolerance: 2
wandb: 	hidden_dims_idx: 2
wandb: 	lr: 0.001
wandb: 	max_abrupt_change_ratio: 0.05
wandb: 	max_sign_changes_ratio: 0.2
wandb: 	max_steps: 100
wandb: 	max_time_detector: 30
wandb: 	reward_dead_band: 0.02
wandb: 	reward_weights_idx: 0
wandb: 	target_update_freq: 100
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.



Episodio 0/300
  Reward: -0.46
  Length: 100
  CTRL Loss: 0.0000
  CTRL Epsilon: 1.0000

Episodio 50/300
  Reward: -1.00
  Length: 100
  CTRL Loss: 9270.6234
  CTRL Epsilon: 0.0831
Evaluación: Reward promedio = -0.83
Agente guardado en: checkpoints/dqn_rare-sweep-1/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 100/300
  Reward: -1.63
  Length: 100
  CTRL Loss: 70677.6965
  CTRL Epsilon: 0.0100
Evaluación: Reward promedio = -1.07
  Sin mejora: 1/10

Episodio 150/300
  Reward: -1.58
  Length: 100
  CTRL Loss: 314901.3143
  CTRL Epsilon: 0.0100
Evaluación: Reward promedio = -0.56
Agente guardado en: checkpoints/dqn_rare-sweep-1/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 200/300
  Reward: -0.85
  Length: 100
  CTRL Loss: 731494.5814
  CTRL Epsilon: 0.0100
Evaluación: Reward promedio = -0.94
  Sin mejora: 1/10

Episodio 250/300
  Reward: -0.68
  Length: 100
  CTRL Loss: 1913217.9477
  CTRL Epsilon: 0.0100
Evaluación: Reward promedio = -1.12
  Sin mejora: 2/10
Run compl

energy,▂███▁▁█████████▇██▃▃████████▇▁█▁▂██▇█▁█▇
epsilon,█▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval_reward,▅▂█▃▁
final_energy_mean10,▁
final_epsilon,▁
final_eval_reward,▁
final_overshoot_mean10,▁
final_reward_mean10,▁
kd_var0,▁▁▁▁▁▃▁▁▁▂▂▄▅▃▃▂▂▂▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▅▁█▁▁
kd_var1,▁▁▁▂▂▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▇▇▁▅▄▇█▁▁▁▁▁▁▂▂███
+8,...


wandb: Agent Starting Run: 5uth2x29 with config:
wandb: 	abrupt_change_threshold: 0.5
wandb: 	batch_size: 32
wandb: 	buffer_size: 50000
wandb: 	buffer_type: simple
wandb: 	delta_percent_ctrl: 0.2
wandb: 	epsilon_decay: 0.9999
wandb: 	error_increase_tolerance: 2
wandb: 	hidden_dims_idx: 2
wandb: 	lr: 0.0001
wandb: 	max_abrupt_change_ratio: 0.03
wandb: 	max_sign_changes_ratio: 0.2
wandb: 	max_steps: 100
wandb: 	max_time_detector: 60
wandb: 	reward_dead_band: 0.02
wandb: 	reward_weights_idx: 1
wandb: 	target_update_freq: 100
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.



Episodio 0/300
  Reward: -0.40
  Length: 100
  CTRL Loss: 315139.1214
  CTRL Epsilon: 0.9931

Episodio 50/300
  Reward: -0.33
  Length: 100
  CTRL Loss: 5028.2094
  CTRL Epsilon: 0.6023
Evaluación: Reward promedio = -0.72
Agente guardado en: checkpoints/dqn_sweet-sweep-2/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 100/300
  Reward: -0.71
  Length: 100
  CTRL Loss: 37263.2868
  CTRL Epsilon: 0.3653
Evaluación: Reward promedio = -0.46
Agente guardado en: checkpoints/dqn_sweet-sweep-2/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 150/300
  Reward: -0.78
  Length: 100
  CTRL Loss: 245437.6727
  CTRL Epsilon: 0.2216
Evaluación: Reward promedio = -2.01
  Sin mejora: 1/10

Episodio 200/300
  Reward: -1.11
  Length: 100
  CTRL Loss: 1210709.6942
  CTRL Epsilon: 0.1344
Evaluación: Reward promedio = -1.12
  Sin mejora: 2/10

Episodio 250/300
  Reward: -2.14
  Length: 100
  CTRL Loss: 2200814.8144
  CTRL Epsilon: 0.0815
Evaluación: Reward promedio = -2.11
  Sin mejora: 3/10
R

...


Episodio 0/300
  Reward: -2.29
  Length: 50
  CTRL Loss: 243583.0683
  CTRL Epsilon: 0.9981

Episodio 50/300
  Reward: -0.06
  Length: 50
  CTRL Loss: 6473.9994
  CTRL Epsilon: 0.7773
Evaluación: Reward promedio = -0.81
Agente guardado en: checkpoints/dqn_azure-sweep-7/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 100/300
  Reward: -0.44
  Length: 50
  CTRL Loss: 102080.5796
  CTRL Epsilon: 0.6054
Evaluación: Reward promedio = -0.39
Agente guardado en: checkpoints/dqn_azure-sweep-7/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 150/300
  Reward: -0.76
  Length: 50
  CTRL Loss: 687058.6516
  CTRL Epsilon: 0.4715
Evaluación: Reward promedio = -1.13
  Sin mejora: 1/10

Episodio 200/300
  Reward: -1.73
  Length: 50
  CTRL Loss: 2227524.5703
  CTRL Epsilon: 0.3672
Evaluación: Reward promedio = -1.25
  Sin mejora: 2/10

Episodio 250/300
  Reward: -0.08
  Length: 50
  CTRL Loss: 4524048.2700
  CTRL Epsilon: 0.2859
Evaluación: Reward promedio = -1.09
  Sin mejora: 3/10
Run co

energy,▇▇▇▁▇▇▇▇▇██▇▇▇▇▇▇█▇▇▇█▇▇▇▇█▇█▇▇▇▇▇▇▇▇▇▇▇
epsilon,█████▇▇▇▇▆▆▆▆▆▆▅▅▄▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
eval_reward,▅█▂▁▂
final_energy_mean10,▁
final_epsilon,▁
final_eval_reward,▁
final_overshoot_mean10,▁
final_reward_mean10,▁
kd_var0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
kd_var1,▁▁▁▁▁▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▅▁▁▁▁▁▁
+8,...


wandb: Agent Starting Run: 3ykum95z with config:
wandb: 	abrupt_change_threshold: 0.2
wandb: 	batch_size: 64
wandb: 	buffer_size: 50000
wandb: 	buffer_type: simple
wandb: 	delta_percent_ctrl: 0.2
wandb: 	epsilon_decay: 0.999
wandb: 	error_increase_tolerance: 1.2
wandb: 	hidden_dims_idx: 0
wandb: 	lr: 0.0001
wandb: 	max_abrupt_change_ratio: 0.1
wandb: 	max_sign_changes_ratio: 0.3
wandb: 	max_steps: 50
wandb: 	max_time_detector: 60
wandb: 	reward_dead_band: 0.02
wandb: 	reward_weights_idx: 3
wandb: 	target_update_freq: 50
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.



Episodio 0/300
  Reward: -0.20
  Length: 50
  CTRL Loss: 0.0000
  CTRL Epsilon: 1.0000

Episodio 50/300
  Reward: -3.54
  Length: 50
  CTRL Loss: 137413.8955
  CTRL Epsilon: 0.0831
Evaluación: Reward promedio = -2.51
Agente guardado en: checkpoints/dqn_morning-sweep-8/agent_ctrl_best.pt
Checkpoint guardado: best

Episodio 100/300
  Reward: -4.71
  Length: 50
  CTRL Loss: 449347.0078
  CTRL Epsilon: 0.0100
Evaluación: Reward promedio = -3.97
  Sin mejora: 1/10


: 

## 6. Visualización local del mejor run


In [ ]:
from Aux.Plots import SimplePlotter, print_summary

# Recuperar el mejor run del sweep
api = wandb.Api()
sweep = api.sweep(f'{WANDB_TEAM}/{WANDB_PROJECT}/{sweep_id}')
best_run = sorted(sweep.runs, key=lambda r: r.summary.get('eval_reward', -float('inf')), reverse=True)[0]

print(f'Mejor run: {best_run.name}')
print(f'eval_reward: {best_run.summary.get("eval_reward"):.4f}')
print(f'Hiperparámetros:')
for k, v in best_run.config.items():
    print(f'  {k}: {v}')